In [1]:
# Setup
import importlib
import os
import subprocess
import sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
REPO_PATH = Path('/content/AI_Agentic_DL') if IS_COLAB else Path.cwd()
TARGET_BRANCH = 'final-pipeline-integration'
REQUIRED_DATA_FILES = ['X_train.npy', 'X_test.npy', 'y_train.npy', 'y_test.npy']
ENV_MARKER = '/tmp/final_pipeline_env_ready_v4'

if IS_COLAB and not REPO_PATH.exists():
    !git clone https://github.com/Lawapaul/AI_Agentic_DL.git /content/AI_Agentic_DL

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

subprocess.check_call(['git', '-C', str(REPO_PATH), 'fetch', 'origin'])
subprocess.check_call(['git', '-C', str(REPO_PATH), 'checkout', TARGET_BRANCH])
subprocess.check_call(['git', '-C', str(REPO_PATH), 'pull', 'origin', TARGET_BRANCH])

def ensure_package(requirement, import_name=None):
    import_name = import_name or requirement.split('==')[0].split('>=')[0].replace('-', '_')
    try:
        importlib.import_module(import_name)
        return False
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', requirement])
        return True

def tensorflow_healthy():
    try:
        subprocess.check_call([
            sys.executable, '-c',
            'import tensorflow as tf; import tensorflow.python; print(tf.__version__)'
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

changed = False
if not os.path.exists(ENV_MARKER):
    changed |= ensure_package('lime>=0.2.0.1', 'lime')
    changed |= ensure_package('transformers==4.41.2', 'transformers')
    changed |= ensure_package('accelerate==0.30.1', 'accelerate')
    changed |= ensure_package('sentencepiece>=0.1.99', 'sentencepiece')
    changed |= ensure_package('networkx>=3.0.0', 'networkx')
    changed |= ensure_package('pyarrow>=15.0.0', 'pyarrow')

if not tensorflow_healthy():
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'tensorflow', 'tensorflow-cpu', 'tf-keras', 'keras', 'ml-dtypes', 'jax', 'jaxlib'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall',
        'numpy==1.26.4',
        'ml-dtypes==0.3.2',
        'tensorflow==2.16.1',
        'keras==3.3.3',
    ])
    changed = True

Path(ENV_MARKER).write_text('ready', encoding='utf-8')
if changed:
    raise SystemExit('Environment updated. Restart runtime once, then rerun this cell.')

for module_name in list(sys.modules):
    if module_name.startswith(('transformers', 'accelerate', 'tokenizers', 'huggingface_hub', 'tensorflow', 'keras', 'jax')):
        del sys.modules[module_name]

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

DEFAULT_PROCESSED_PATHS = [
    '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed',
    '/content/drive/MyDrive/AI Agentic/data/processed',
    '/content/AI_Agentic_DL/data/processed',
    str(REPO_PATH / 'data' / 'processed'),
]
DEFAULT_MODEL_PATHS = [
    '/content/drive/MyDrive/Deep Learning Project/AI Agentic/saved_models/hybrid_cnn_lstm_full_2_2m.keras',
    '/content/drive/MyDrive/Deep Learning Project/AI Agentic/saved_models/ids_model.keras',
    '/content/drive/MyDrive/AI Agentic/saved_models/hybrid_cnn_lstm_full_2_2m.keras',
    '/content/AI_Agentic_DL/saved_models/ids_model.keras',
    str(REPO_PATH / 'saved_models' / 'ids_model.keras'),
]

def first_existing_processed(paths):
    for path in paths:
        path_obj = Path(path)
        if path_obj.exists() and all((path_obj / name).exists() for name in REQUIRED_DATA_FILES):
            return str(path_obj)
    return None

def first_existing_file(paths):
    for path in paths:
        if Path(path).exists():
            return path
    return None

PROCESSED_PATH = first_existing_processed(DEFAULT_PROCESSED_PATHS)
MODEL_PATH = first_existing_file(DEFAULT_MODEL_PATHS)

if PROCESSED_PATH is None:
    raise FileNotFoundError('Could not find processed dataset files. Checked: ' + ', '.join(DEFAULT_PROCESSED_PATHS))

if MODEL_PATH is None:
    raise FileNotFoundError('Could not find a trained hybrid model. Checked: ' + ', '.join(DEFAULT_MODEL_PATHS))

print('Repo branch:', TARGET_BRANCH)
print('Processed path:', PROCESSED_PATH)
print('Model path:', MODEL_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo branch: final-pipeline-integration
Processed path: /content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed
Model path: /content/drive/MyDrive/Deep Learning Project/AI Agentic/saved_models/hybrid_cnn_lstm_full_2_2m.keras


In [2]:
# Load modules
import importlib
import sys

importlib.invalidate_caches()
if 'integration.final_pipeline' in sys.modules:
    del sys.modules['integration.final_pipeline']

import integration.final_pipeline as final_pipeline
final_pipeline = importlib.reload(final_pipeline)
run_demo_pipeline = final_pipeline.run_demo_pipeline
run_full_pipeline = final_pipeline.run_full_pipeline


In [3]:
# Run full pipeline (safe mode)
full_summary = run_full_pipeline(PROCESSED_PATH, MODEL_PATH)
print(full_summary['status'])
print(full_summary['runtime'])
print(full_summary['decision_planner'])
print(full_summary['retraining'])


INFO:integration.final_pipeline:Starting final pipeline with runtime={'available_ram_gb': 11.26, 'max_samples': 50, 'batch_size': 64, 'device': 'CPU', 'demo_mode': False, 'platform': 'Linux-6.6.113+-x86_64-with-glibc2.35'}


Loading model from: /content/drive/MyDrive/Deep Learning Project/AI Agentic/saved_models/hybrid_cnn_lstm_full_2_2m.keras

=== Initializing Feature Gradient Explainer ===
Number of features: 78


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


completed
{'available_ram_gb': 11.26, 'max_samples': 50, 'batch_size': 64, 'device': 'CPU', 'demo_mode': False, 'platform': 'Linux-6.6.113+-x86_64-with-glibc2.35'}
{'best_planner': 'policy', 'metrics': {'rule_based': {'accuracy': 0.6, 'consistency': 1.0, 'reward': 0.23538, 'majority_action': 'Block'}, 'confidence_based': {'accuracy': 0.04, 'consistency': 1.0, 'reward': -1.04058, 'majority_action': 'No Action'}, 'risk_based': {'accuracy': 0.04, 'consistency': 1.0, 'reward': -1.04058, 'majority_action': 'No Action'}, 'hybrid': {'accuracy': 0.06, 'consistency': 1.0, 'reward': -0.99682, 'majority_action': 'No Action'}, 'rl': {'accuracy': 0.58, 'consistency': 1.0, 'reward': 0.16098, 'majority_action': 'Alert'}, 'policy': {'accuracy': 0.84, 'consistency': 1.0, 'reward': 0.7664200000000001, 'majority_action': 'Block'}, 'llm': {'accuracy': 0.0, 'consistency': 1.0, 'reward': -1.1288600000000002, 'majority_action': 'No Action'}}}
{'best_method': 'rl_trainer', 'metrics': {'rl_trainer': {'accuracy

In [4]:
# Run demo pipeline (presentation mode)
demo_summary = run_demo_pipeline()
print('Demo samples:', len(demo_summary.get('records', [])))


INFO:integration.final_pipeline:Starting final pipeline with runtime={'available_ram_gb': 11.01, 'max_samples': 8, 'batch_size': 64, 'device': 'CPU', 'demo_mode': True, 'platform': 'Linux-6.6.113+-x86_64-with-glibc2.35'}


Loading model from: /content/drive/MyDrive/Deep Learning Project/AI Agentic/saved_models/hybrid_cnn_lstm_full_2_2m.keras

=== Initializing Feature Gradient Explainer ===
Number of features: 78


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


---------------------------------
 SAMPLE 1
---------------------------------
Classifier Output: {'prediction': 0, 'confidence': 0.999997079372406}
Top Features: ['feature_70 (-0.0008)', 'feature_72 (-0.0007)', 'feature_40 (-0.0003)', 'feature_53 (-0.0003)', 'feature_1 (0.0003)', 'feature_7 (-0.0003)', 'feature_20 (0.0002)', 'feature_38 (0.0002)']
Graph Correlation: {'top_neighbors': [{'sample_id': 'sample_-1', 'weight': 0.5}], 'max_weight': 0.5}
Memory Match: {'labels': [0], 'similarities': [0.5], 'top_similarity': 0.5}
Fused Risk Score: {'risk_score': 0.6750014199533325, 'severity': 'HIGH'}
LLM Reasoning: High risk detected from correlated graph behavior, memory similarity, and the most important network features. Recommended action: Block.
Decision: {'event_id': 'sample_0', 'risk_score': 0.6750014199533325, 'uncertainty': 2.9206275939941406e-06, 'explanation': 'High risk detected from correlated graph behavior, memory similarity, and the most important network features. Recommended 

In [5]:
import subprocess, textwrap
subprocess.run(["git", "-C", "/content/AI_Agentic_DL", "rev-parse", "--short", "HEAD"], check=True)
print(open("/content/AI_Agentic_DL/llm_reasoning/llm_pipeline.py").read()[0:1200])
print(open("/content/AI_Agentic_DL/integration/final_pipeline.py").read()[640:900])


"""End-to-end lightweight LLM reasoning pipeline."""

from llm_reasoning.llm_engine import LLMReasoningEngine
from llm_reasoning.reasoning_generator import ReasoningGenerator


LABEL_MAP = {
    0: "Normal Traffic",
    1: "DoS Attack",
    2: "Port Scan",
    3: "Brute Force Attack",
    4: "Web Attack",
    5: "Botnet",
    6: "Infiltration",
}


class LLMPipeline:
    """Generate prompts and model reasoning for one IDS sample."""

    def __init__(self, model: str = "google/flan-t5-small"):
        self.engine = LLMReasoningEngine()
        self.generator = ReasoningGenerator(model=model)

    def run(
        self,
        features,
        prediction,
        confidence: float,
        risk_score: float | None = None,
        top_features=None,
        memory_similarity: float | None = None,
        graph_weight: float | None = None,
        decision: str | None = None,
    ):
        attack_name = LABEL_MAP.get(int(prediction), "Unknown Attack")
        top_feature_names = [str(v